In [ ]:
import time
import random
import math
from collections import deque

def in_mt(mt):
    res = ""
    for i in range(3):
        row = ""
        for j in range(3):
            val = mt[i*3 + j]
            if val == 0:
                row += " [ ] "
            else:
                row += f"  {val}  "
        res += row + "\n"
    res += "-" * 20 + "\n"
    return res

def get_successors(mt):
    pos = mt.index(0)
    r, c = pos // 3, pos % 3
    successors = []
    
    def swap(mt, i, j):
        new_mt = list(mt)
        new_mt[i], new_mt[j] = new_mt[j], new_mt[i]
        return new_mt, new_mt[i]
        
    if c > 0: 
        new_state, cost = swap(mt, pos, pos - 1)
        successors.append(("Trái", new_state, cost))
    if c < 2: 
        new_state, cost = swap(mt, pos, pos + 1)
        successors.append(("Phải", new_state, cost))
    if r > 0: 
        new_state, cost = swap(mt, pos, pos - 3)
        successors.append(("Lên", new_state, cost))
    if r < 2: 
        new_state, cost = swap(mt, pos, pos + 3)
        successors.append(("Xuống", new_state, cost))
    return successors

def transition_state(state, action):
    pos = state.index(0)
    r, c = pos // 3, pos % 3
    target_pos = pos
    if action == "Lên" and r > 0:
        target_pos = pos - 3
    elif action == "Xuống" and r < 2:
        target_pos = pos + 3
    elif action == "Trái" and c > 0:
        target_pos = pos - 1
    elif action == "Phải" and c < 2:
        target_pos = pos + 1
        
    if target_pos == pos:
        return state
        
    new_state = list(state)
    new_state[pos], new_state[target_pos] = new_state[target_pos], new_state[pos]
    return tuple(new_state)

def transition_belief_state(belief_state, action):
    return frozenset(transition_state(s, action) for s in belief_state)

def count_inversions(state):
    # Đếm số phép nghịch thế để kiểm tra tính giải được của trạng thái
    arr = [val for val in state if val != 0]
    inv = 0
    for i in range(len(arr)):
        for j in range(i + 1, len(arr)):
            if arr[i] > arr[j]:
                inv += 1
    return inv

def conformant_bfs_solve(start_states, goal_state):
    """
    Thuật toán Conformant Search sử dụng BFS với Early Goal Test.
    - start_states: Danh sách các trạng thái bắt đầu [[...], [...]]
    - goal_state: Trạng thái đích [...]
    """
    initial_belief = frozenset(tuple(s) for s in start_states)
    goal_tuple = tuple(goal_state)
    
    # Kiểm tra chẵn lẻ nghịch thế để cảnh báo sớm nếu không giải được
    goal_inv = count_inversions(goal_tuple) % 2
    unsolvable_states = []
    for s in initial_belief:
        if count_inversions(s) % 2 != goal_inv:
            unsolvable_states.append(list(s))
            
    log_data = []
    
    if unsolvable_states:
        fail_html = f"<b style='color: #c53030;'>PHÂN TÍCH KHÔNG THỂ GIẢI ĐƯỢC ❌</b><br>"
        fail_html += f"Trạng thái đích {list(goal_tuple)} có độ chẵn lẻ nghịch thế là {goal_inv}.<br>"
        fail_html += "Các trạng thái ban đầu sau đây có độ chẵn lẻ nghịch thế khác đích nên <b>không thể đạt đích</b>:<br>"
        for us in unsolvable_states:
            fail_html += f"👉 State {us} (nghịch thế = {count_inversions(us)})<br>"
        fail_html += "➔ Do đó, <b>không tồn tại bất kỳ kế hoạch Conformant nào</b> để giải quyết đồng thời!"
        log_data.append({
            "step": "KQ",
            "action_html": fail_html,
            "frontier_str": "Không giải được!",
            "reached_str": "Thất bại"
        })
        return None, 0, log_data
        
    if len(initial_belief) == 1 and next(iter(initial_belief)) == goal_tuple:
        log_data.append({
            "step": 0,
            "action_html": "Trạng thái ban đầu đã khớp với Đích!",
            "frontier_str": "Đã đạt đích!",
            "reached_str": "Thành công"
        })
        return [], 1, log_data
        
    frontier = deque([(initial_belief, [])])
    explored = {initial_belief}
    nodes_generated = 1
    
    log_data.append({
        "step": 0,
        "action_html": f"<b>Khởi tạo Conformant Search (BFS Early)</b> với {len(initial_belief)} trạng thái:<br>" +
                       "<br>".join([f"Trạng thái {i+1}: {list(s)}" for i, s in enumerate(initial_belief)]),
        "frontier_str": "Frontier: 1",
        "reached_str": "Explored: 1"
    })
    
    actions = ["Lên", "Xuống", "Trái", "Phải"]
    iteration = 0
    max_iterations = 200000
    
    while frontier and iteration < max_iterations:
        iteration += 1
        belief_node, path = frontier.popleft()
        
        for action in actions:
            child_belief = transition_belief_state(belief_node, action)
            nodes_generated += 1
            
            if len(child_belief) == 1 and next(iter(child_belief)) == goal_tuple:
                action_html = f"<b>Bước lặp {iteration}</b> - Hành động: <b>{action}</b><br>" 
                action_html += f"👉 <b>ĐẠT ĐÍCH THÀNH CÔNG!</b><br>" 
                action_html += f"Mọi trạng thái đã hội tụ về Đích: {list(goal_tuple)}"
                
                log_data.append({
                    "step": len(path) + 1,
                    "action_html": action_html,
                    "frontier_str": "Đạt đích!",
                    "reached_str": f"Tổng nút sinh: {nodes_generated}"
                })
                return path + [(action, child_belief)], nodes_generated, log_data
                
            if child_belief not in explored:
                explored.add(child_belief)
                frontier.append((child_belief, path + [(action, child_belief)]))
                
                states_str = "<br>".join([f"&nbsp;&nbsp;+ {list(s)}" for s in child_belief])
                action_html = f"<b>Bước lặp {iteration}</b> - Hành động: <b>{action}</b><br>" 
                action_html += f"Belief State mới ({len(child_belief)} trạng thái):<br>{states_str}"
                
                log_data.append({
                    "step": len(path) + 1,
                    "action_html": action_html,
                    "frontier_str": f"Frontier: {len(frontier)}",
                    "reached_str": f"Explored: {len(explored)}"
                })
                
    fail_html = f"<b style='color: #c53030;'>Không tìm thấy giải pháp sau khi duyệt toàn bộ không gian trạng thái!</b>"
    log_data.append({
        "step": "KQ",
        "action_html": fail_html,
        "frontier_str": "Thất bại",
        "reached_str": f"Nodes: {nodes_generated}"
    })
    return None, nodes_generated, log_data
